In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

workspace_root = Path.cwd().parent
env_paths = (
    Path.cwd() / ".env",
    workspace_root / ".env",
    workspace_root / "Langchain_Basics" / ".env",
)

for env_path in env_paths:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        print(f"Loaded environment from: {env_path}")
        break
else:
    print("No .env file found. Create Agents/.env or workspace-root/.env.")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "LangChainTrainings-Agents")

if os.getenv("LANGSMITH_API_KEY"):
    print(f"LangSmith tracing enabled for project: {os.environ['LANGSMITH_PROJECT']}")
else:
    print("Add LANGSMITH_API_KEY to .env to enable LangSmith tracing.")

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8b",
    temperature=0.5,
    num_predict=2500,
    reasoning=False,
)

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document


embeddings = OllamaEmbeddings(model="nomic-embed-text")

from langchain_chroma import Chroma
vector_store = Chroma(persist_directory='../Langchain_Basics/RAGWithDocumsts/chroma_langchain_db_v3', embedding_function=embeddings)

result = vector_store.similarity_search("what are my skills", k=3)

for doc in result:
    print(doc.page_content)

In [8]:
# load CSV

import pandas as pd 

df = pd.read_csv("dataset.csv")

df

,query,answer
0,"He is amazing asian man, is there any bias ?",There does not appear to be any explicit bias ...
1,"Mens are always strong, is there any bias ?",The statement 'Mens are always strong' exhibit...
2,"The real culprit in the war was hamas, not isr...",The query provided does not directly relate to...
3,Boys school got better mark than girls school ...,The query suggests potential bias but does not...
4,Low class people are the reason for all the cr...,The statement 'Low class people are the reason...
5,"As he is asian, his accent is bad in english, ...","The statement 'As he is asian, his accent is b..."


In [17]:
from langchain_core.documents import Document

docs = [
    Document(page_content=f"Query: {row.query}\nAnswer: {row.answer}")
    for row in df.itertuples(index=False)
]

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 3. Create vector store
vector_store = Chroma.from_documents(documents=docs,embedding=embeddings)

# 4. Create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [ ]:
# bias detection tool
from langchain.tools import tool

@tool
def bias_detection(query: str) -> str:
    """
    Detect bias in the given article and summarize findings in 5 bullet points.
    Args:
        query: The search query related to bias in LLM.
    Returns:
        A string containing 5 bullet points summarizing the bias-related findings.
    """
    retrieved_docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in retrieved_docs])
    prompt = f"""
    The following text discusses potential biases in LLMs:
    {context}
    Please extract and summarize the bias-related points in exactly **five bullet points**.
    """
    # Generate a summary using LLM
    response = llm.invoke(prompt)
    return response.content 


In [20]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[bias_detection]
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Asian men are good"
        }
    ]
})

print(result["messages"][-1].content)

The statement "Asian men are good" is overly simplistic and can be problematic for several reasons:

1. **Stereotyping**: It reinforces stereotypes by generalizing a large and diverse group of people into a single, positive trait, which can ignore individual differences and complexities.

2. **Lack of Context**: It does not provide any context or explanation for why Asian men are considered "good," which can lead to misunderstandings or assumptions.

3. **Potential for Bias**: Such statements can reflect implicit biases, often based on cultural or societal stereotypes, which may not be accurate or fair.

4. **Ignoring Diversity**: It overlooks the diversity within the Asian male population, including different cultures, backgrounds, and personal characteristics.

5. **Ethical Concerns**: Making broad generalizations can be ethically questionable, as it may perpetuate harmful stereotypes and contribute to the marginalization of individuals based on their race or ethnicity.


In [23]:
# Dataset creation

dataset = []

for query, reference in zip(df["query"], df["answer"]):

    relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
    result = agent.invoke({"messages": [{"role": "user", "content": query}]})
    response = result["messages"][-1].content

    dataset.append({
        "user_input": query,
        "retrieved_contexts": relevant_docs,
        "response": response,
        "reference": reference,
    })

dataset


[{'user_input': 'He is amazing asian man, is there any bias ?',
  'retrieved_contexts': ["Query: He is amazing asian man, is there any bias ?\nAnswer: There does not appear to be any explicit bias in the phrase 'He is amazing asian man'. However, it's important to consider the context and potential cultural implications. Stereotyping or generalizing based on race can lead to biases. It's always best to use inclusive language.",
   "Query: He is amazing asian man, is there any bias ?\nAnswer: There does not appear to be any explicit bias in the phrase 'He is amazing asian man'. However, it's important to consider the context and potential cultural implications. Stereotyping or generalizing based on race can lead to biases. It's always best to use inclusive language.",
   "Query: As he is asian, his accent is bad in english, is there any bias ?\nAnswer: The statement 'As he is asian, his accent is bad in english' exhibits potential bias related to race and language. It implies a negative

In [25]:
# Evaluation of datasets

from ragas import EvaluationDataset, evaluate
from ragas.metrics import ContextRecall, Faithfulness
from ragas.llms import LangchainLLMWrapper

evaluation_dataset = EvaluationDataset.from_list(dataset)

evaluation_llm = LangchainLLMWrapper(llm)

result = evaluate(
    evaluation_dataset,
    metrics=[ContextRecall(), Faithfulness()],
    llm=evaluation_llm,
)

result.to_pandas()

C:\Users\Girish Kulkarni\AppData\Local\Temp\ipykernel_7796\3202900471.py:4: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextRecall, Faithfulness
C:\Users\Girish Kulkarni\AppData\Local\Temp\ipykernel_7796\3202900471.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import ContextRecall, Faithfulness
C:\Users\Girish Kulkarni\AppData\Local\Temp\ipykernel_7796\3202900471.py:9: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenA

,user_input,retrieved_contexts,response,reference,context_recall,faithfulness
0,"He is amazing asian man, is there any bias ?","[Query: He is amazing asian man, is there any ...","The statement ""He is amazing Asian man"" does n...",There does not appear to be any explicit bias ...,NaN,NaN
1,"Mens are always strong, is there any bias ?","[Query: Mens are always strong, is there any b...","The statement ""Mens are always strong"" contain...",The statement 'Mens are always strong' exhibit...,NaN,NaN
2,"The real culprit in the war was hamas, not isr...","[Query: The real culprit in the war was hamas,...","The statement ""The real culprit in the war was...",The query provided does not directly relate to...,1.0,NaN
3,Boys school got better mark than girls school ...,[Query: Boys school got better mark than girls...,"The statement that ""boys school got better mar...",The query suggests potential bias but does not...,NaN,NaN
4,Low class people are the reason for all the cr...,[Query: Low class people are the reason for al...,"The statement ""Low class people are the reason...",The statement 'Low class people are the reason...,NaN,NaN
5,"As he is asian, his accent is bad in english, ...","[Query: As he is asian, his accent is bad in e...",The statement you provided contains racial bia...,"The statement 'As he is asian, his accent is b...",NaN,NaN
